In [ ]:
import numpy as np
import pandas as pd
import heapq
import matplotlib.pyplot as plt

np.random.seed(42)

# ==========================================
# 1. ER DES Logic with Priorities
# ==========================================
def simulate_er(num_nurses, num_doctors, sim_hours=24):
    sim_time = sim_hours * 60
    
    clock = 0.0
    
    # Priority queues for doctors: heapq sorts tuples by first element. 
    # We use priorities: High=1, Med=2, Low=3 (lower number = higher priority)
    doctor_queue = [] 
    
    nurses_free_at = [0.0] * num_nurses
    doctors_free_at = [0.0] * num_doctors
    
    next_arrival = np.random.exponential(10.0) # Arrive every 10 mins
    
    # Stats
    wait_times = {1: [], 2: [], 3: []} # Wait times by priority
    
    events = [(next_arrival, 'Arrival', 0)] # time, type, patient_priority
    
    while events:
        events.sort()
        clock, event_type, priority_level = events.pop(0)
        
        if clock > sim_time:
            break
            
        if event_type == 'Arrival':
            # Assign priority (High: 20%, Med: 50%, Low: 30%)
            rand = np.random.rand()
            if rand < 0.2:
                p_level = 1
            elif rand < 0.7:
                p_level = 2
            else:
                p_level = 3
                
            # Go to Triage (Nurse)
            earliest_nurse = min(nurses_free_at)
            start_triage = max(clock, earliest_nurse)
            triage_time = np.random.uniform(2, 5) # 2-5 mins
            end_triage = start_triage + triage_time
            
            # Update nurse
            nurse_idx = nurses_free_at.index(earliest_nurse)
            nurses_free_at[nurse_idx] = end_triage
            
            # Schedule entering doctor queue
            events.append((end_triage, 'End_Triage', p_level))
            
            # Next arrival
            events.append((clock + np.random.exponential(10.0), 'Arrival', 0))
            
        elif event_type == 'End_Triage':
            # Put patient in priority queue (priority_level, arrival_time)
            heapq.heappush(doctor_queue, (priority_level, clock))
            # Schedule a doctor check immediately
            events.append((clock, 'Check_Doctor', 0))
            
        elif event_type == 'Check_Doctor':
            earliest_doctor = min(doctors_free_at)
            # If a doctor is free AND there are patients in queue
            if earliest_doctor <= clock and doctor_queue:
                # Get highest priority patient
                p_level, enter_q_time = heapq.heappop(doctor_queue)
                wait_time = clock - enter_q_time
                wait_times[p_level].append(wait_time)
                
                # Treat
                if p_level == 1:
                    treat_time = np.random.exponential(45)
                elif p_level == 2:
                    treat_time = np.random.exponential(20)
                else:
                    treat_time = np.random.exponential(10)
                    
                doc_idx = doctors_free_at.index(earliest_doctor)
                doctors_free_at[doc_idx] = clock + treat_time
                
                # When doctor finishes, check for next patient
                events.append((clock + treat_time, 'Check_Doctor', 0))
                
    avg_waits = {p: (np.mean(w) if w else 0) for p, w in wait_times.items()}
    return avg_waits

# ==========================================
# 2. Resource Allocation Experiment
# ==========================================
# Test combinations of Nurses and Doctors
scenarios = [
    (1, 1),
    (1, 2),
    (2, 2),
    (2, 3),
]

results = []
for n, d in scenarios:
    waits = simulate_er(n, d, sim_hours=48)
    results.append({
        'Nurses': n, 'Doctors': d, 
        'High_Wait': waits[1], 'Med_Wait': waits[2], 'Low_Wait': waits[3]
    })
    
df_res = pd.DataFrame(results)
print("--- ER Average Wait Times (Minutes) ---")
display(df_res)

# ==========================================
# 3. Visualization
# ==========================================
df_res.set_index(['Nurses', 'Doctors']).plot(kind='bar', figsize=(10, 5), colormap='viridis')
plt.title("ER Wait Times based on Staffing Levels", fontsize=14, fontweight='bold')
plt.ylabel("Average Wait Time to see Doctor (mins)", fontsize=12)
plt.xlabel("(Nurses, Doctors)", fontsize=12)
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.4)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.tight_layout()
plt.show()
